# 🍽 Restaurant Recommendation System
## Content-Based Filtering using TF-IDF & Cosine Similarity
**Dataset:** Zomato Bangalore Restaurants  
**Objective:** Build a restaurant recommendation system using Content-Based Filtering


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import plotly.offline as py
import plotly.graph_objs as go
import seaborn as sns
import warnings
import nltk
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import flask
from flask import Flask, render_template, request
import pickle

warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

nltk.download('stopwords')
print('Libraries loaded successfully!')

## 2. Load & Inspect Dataset

In [ ]:
# Load the Zomato Bangalore dataset
df = pd.read_csv('../Dataset/zomato.csv', encoding='latin-1')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
# Dataset info
df.info()

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Missing values
print('Missing Values:')
print(df.isnull().sum())

## 3. Data Pre-Processing

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
print('Columns:', list(df.columns))

# Rename cost column
cost_col = [c for c in df.columns if 'cost' in c or 'approx' in c]
if cost_col:
    df.rename(columns={cost_col[0]: 'cost'}, inplace=True)

# Clean rate column - remove '/5' suffix
df['rate'] = df['rate'].astype(str)
df['rate'] = df['rate'].apply(lambda x: x.replace('/5', '').strip() if '/5' in x else x)
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')

# Clean votes
df['votes'] = pd.to_numeric(df['votes'], errors='coerce')

# Clean cost
df['cost'] = df['cost'].astype(str).str.replace(',', '').str.strip()
df['cost'] = pd.to_numeric(df['cost'], errors='coerce')

# Remove duplicates
df.drop_duplicates(inplace=True)

# Fill missing values
df['cuisines'] = df['cuisines'].fillna('Unknown')
df['rate'] = df['rate'].fillna(df['rate'].median())
df['votes'] = df['votes'].fillna(0)
df['cost'] = df['cost'].fillna(df['cost'].median())

# Convert cost to thousands
df['cost'] = (df['cost'] / 1000).round(1)

print(f'\nCleaned dataset: {df.shape}')
df.head()

In [ ]:
# Compute mean rating per restaurant
rating_df = df.groupby('name').agg({'rate': 'mean', 'votes': 'sum'}).reset_index()
rating_df.columns = ['name', 'mean_rating', 'total_votes']
rating_df['mean_rating'] = rating_df['mean_rating'].round(2)

df = df.merge(rating_df, on='name', how='left')
print('Mean ratings computed!')
df[['name', 'cuisines', 'mean_rating', 'cost']].head(10)

## 4. Exploratory Data Analysis & Visualization

In [ ]:
# Top 15 Most Popular Cuisines
all_cuisines = []
for c in df['cuisines'].dropna():
    all_cuisines.extend([x.strip() for x in str(c).split(',')])
cuisine_counts = pd.Series(all_cuisines).value_counts().head(15)

plt.figure(figsize=(12, 7))
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, 15))
bars = plt.barh(cuisine_counts.index[::-1], cuisine_counts.values[::-1], color=colors[::-1])
plt.title('Top 15 Most Popular Cuisines in Bangalore', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Number of Restaurants', fontsize=12)
plt.tight_layout()
plt.savefig('../Flask/static/plots/top_cuisines.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Rating Distribution
plt.figure(figsize=(10, 6))
plt.hist(df['rate'].dropna(), bins=30, color='#c8401a', edgecolor='white', alpha=0.85)
plt.title('Restaurant Rating Distribution', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Rating (out of 5)', fontsize=12)
plt.ylabel('Number of Restaurants', fontsize=12)
plt.axvline(df['rate'].mean(), color='#d4a017', linestyle='--', linewidth=2, label=f'Mean: {df["rate"].mean():.2f}')
plt.legend()
plt.tight_layout()
plt.savefig('../Flask/static/plots/rating_dist.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cost Distribution
plt.figure(figsize=(10, 6))
plt.hist(df['cost'].dropna(), bins=30, color='#2ecc71', edgecolor='white', alpha=0.85)
plt.title('Cost Distribution (₹ in thousands for two)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Cost (₹k)', fontsize=12)
plt.ylabel('Number of Restaurants', fontsize=12)
plt.tight_layout()
plt.savefig('../Flask/static/plots/cost_dist.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Rating vs Cost Scatter Plot
plt.figure(figsize=(10, 7))
plt.scatter(df['cost'], df['rate'], alpha=0.4, c='#9b59b6', s=15, edgecolors='none')
plt.title('Rating vs Cost', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Cost (₹k for two)', fontsize=12)
plt.ylabel('Rating', fontsize=12)
plt.tight_layout()
plt.savefig('../Flask/static/plots/rating_vs_cost.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 Restaurants by Rating
top_restaurants = df.groupby('name')['rate'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
colors = plt.cm.Reds(np.linspace(0.5, 0.9, len(top_restaurants)))
plt.bar(range(len(top_restaurants)), top_restaurants.values, color=colors[::-1])
plt.xticks(range(len(top_restaurants)), top_restaurants.index, rotation=30, ha='right', fontsize=10)
plt.title('Top 10 Highest Rated Restaurants', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Mean Rating', fontsize=12)
plt.ylim(3.5, 5.0)
plt.tight_layout()
plt.savefig('../Flask/static/plots/top_restaurants.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Content-Based Filtering
### Merging Datasets & Creating the Recommender

In [ ]:
# Create a unique restaurants dataframe (one row per restaurant)
restaurant_df = df.drop_duplicates(subset='name').copy()
restaurant_df = restaurant_df.reset_index(drop=True)

print(f'Unique restaurants: {len(restaurant_df)}')
restaurant_df[['name', 'cuisines', 'mean_rating', 'cost']].head()

In [ ]:
# TF-IDF Vectorization on cuisines
tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=0.0,
    stop_words='english'
)

# Clean cuisines text
restaurant_df['cuisines_clean'] = (
    restaurant_df['cuisines'].fillna('').str.lower().str.replace(',', ' ')
)

# Fit and transform
tfidf_matrix = tfidf.fit_transform(restaurant_df['cuisines_clean'])

print(f'TF-IDF Matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

In [ ]:
# Compute Cosine Similarity Matrix
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
print(f'Cosine Similarity Matrix shape: {cosine_sim.shape}')
print('Sample similarity scores (first 5x5):')
print(np.round(cosine_sim[:5, :5], 3))

In [ ]:
# Create reverse mapping of restaurant name -> index
indices = pd.Series(restaurant_df.index, index=restaurant_df['name']).drop_duplicates()
print(f'Index mapping created for {len(indices)} restaurants')

In [ ]:
# Recommendation Function
def get_recommendations(name, cosine_sim=cosine_sim, n=10):
    """
    Get top N restaurant recommendations based on content-based filtering.
    
    Parameters:
        name (str): Restaurant name to base recommendations on
        cosine_sim: Precomputed cosine similarity matrix
        n (int): Number of recommendations
    
    Returns:
        DataFrame with recommended restaurants
    """
    try:
        # Get the index of the restaurant
        idx = indices[name]
        
        # Get pairwise cosine similarity scores
        sim_scores = list(enumerate(cosine_sim[idx]))
        
        # Sort restaurants by similarity score
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # Get scores of top N most similar restaurants (excluding itself)
        sim_scores = sim_scores[1:n+1]
        
        # Get restaurant indices
        rest_indices = [i[0] for i in sim_scores]
        
        # Return recommendations with relevant columns
        result = restaurant_df[['name', 'cuisines', 'mean_rating', 'cost']].iloc[rest_indices]
        return result
    
    except KeyError:
        return f"Restaurant '{name}' not found in dataset!"

## 6. Predicting Results

In [ ]:
# Test: Get recommendations for 'Jalsa'
print('=== Recommendations for: Jalsa ===')
jalsa_recs = get_recommendations('Jalsa')
print(jalsa_recs.to_string(index=False))

In [ ]:
# Test: Get recommendations for 'Cinnamon'
print('=== Recommendations for: Cinnamon ===')
cinnamon_recs = get_recommendations('Cinnamon')
print(cinnamon_recs.to_string(index=False))

## 7. Save Model

In [ ]:
import os
os.makedirs('../Flask', exist_ok=True)

# Save model as pickle file
model_data = {
    'df': restaurant_df,
    'cosine_sim': cosine_sim,
    'indices': indices,
    'tfidf': tfidf
}

with open('../Flask/restaurant.pkl', 'wb') as f:
    pickle.dump(model_data, f)

# Save processed data
restaurant_df.to_csv('../Flask/restaurant1.csv', index=False)

print('Model saved to: ../Flask/restaurant.pkl')
print('Data saved to: ../Flask/restaurant1.csv')
print(f'Total restaurants in model: {len(restaurant_df)}')

## 8. Model Accuracy / Evaluation

For content-based filtering systems, we evaluate quality using:
- **Precision**: How many recommended items are relevant
- **Coverage**: What percentage of the catalog can be recommended
- **Diversity**: How different are the recommendations

In [ ]:
# Evaluate: Check recommendation coverage
total_restaurants = len(restaurant_df)

# Test a sample of restaurants
sample_names = restaurant_df['name'].sample(min(20, total_restaurants)).tolist()

coverage_count = 0
for name in sample_names:
    recs = get_recommendations(name)
    if isinstance(recs, pd.DataFrame) and len(recs) > 0:
        coverage_count += 1

coverage = (coverage_count / len(sample_names)) * 100
print(f'Model Coverage: {coverage:.1f}%')

# Average cuisine similarity in recommendations
test_name = sample_names[0]
test_cuisines = set(restaurant_df[restaurant_df['name'] == test_name]['cuisines'].iloc[0].split(','))
recs = get_recommendations(test_name)

if isinstance(recs, pd.DataFrame):
    similarity_scores = []
    for _, row in recs.iterrows():
        rec_cuisines = set(row['cuisines'].split(','))
        intersection = len(test_cuisines & rec_cuisines)
        union = len(test_cuisines | rec_cuisines)
        jaccard = intersection / union if union > 0 else 0
        similarity_scores.append(jaccard)
    
    avg_similarity = np.mean(similarity_scores)
    print(f'Average Cuisine Similarity (Jaccard): {avg_similarity:.3f} ({avg_similarity*100:.1f}%)')

print(f'\nModel Summary:')
print(f'  Total Restaurants: {total_restaurants}')
print(f'  TF-IDF Features: {len(tfidf.vocabulary_)}')
print(f'  Similarity Matrix: {cosine_sim.shape}')
print(f'  Coverage: {coverage:.1f}%')

## 9. Flask Application
Run the Flask app from the command line:
```bash
cd Flask
python app1.py
```
Then open: http://127.0.0.1:5000